<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop_DHS_2026/blob/main/05_movie_recommendation_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Personalized Movie Recommendation
### Neo4j GraphRAG (HybridRetriever) + Graphiti Memory + ReAct Agent

Single user demo (`user_1`). Functions only, minimal setup.

In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 44 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 2.03 MiB | 4.93 MiB/s, done.
Resolving deltas: 100% (19/19), done.


## Install Required Packages

In [2]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 81.6 MB/s eta 0:00:00


## Credentials & Drivers
The Neo4j driver allows you to connect and perform read and write transactions with the database.

In [4]:
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---
os.environ.setdefault(
    'OPENAI_API_KEY',
    os.getenv('OPENAI_API_KEY')
)

print('NEO4J_URI :', NEO4J_URI)
print('OPENAI key set:', bool(os.environ.get('OPENAI_API_KEY')))

NEO4J_URI : bolt://32.192.25.99
OPENAI key set: True


In [5]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j ✔')

Connected to Neo4j ✔


## Imports

In [7]:
!pip install graphiti_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.1/396.1 kB 18.9 MB/s eta 0:00:00


In [14]:
# 1. Imports
import os
import re
import asyncio
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv

from neo4j import GraphDatabase

from neo4j_graphrag.retrievers import HybridRetriever
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings as GraphRAGEmbeddings

from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType

In [15]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

# Main LLM used by LLMEntityRelationExtractor
llm = OpenAILLM(
    model_name='gpt-5-mini',
)

embedder = OpenAIEmbeddings(model='text-embedding-3-small')
print('LLM + embedder ready ✔')

VECTOR_INDEX = "movie_embedding_index"
FULLTEXT_INDEX = "movie_text_index"

USER_ID = "user_1"

# simple global holders (functions only, no classes)
STATE = {}

LLM + embedder ready ✔


In [16]:
# 3. Neo4j setup
def setup_neo4j():
    # connect to the existing movie knowledge graph
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()
    STATE["driver"] = driver
    print("Neo4j connected.")
    return driver

In [22]:
!pip install langchain-classic

In [24]:
from langchain_classic.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    PromptTemplate,
)
import re
import asyncio
from typing import List, Union
from langchain_classic.agents import AgentOutputParser, AgentExecutor, create_react_agent
from langchain_classic.schema import AgentAction, AgentFinish



# 1️⃣  ───── Robust multi-action parser ────────────────────────────
class MultiActionOutputParser(AgentOutputParser):
    """
    Extract *all* Action / Action Input pairs, even when inputs are
    multi-line or there is text between blocks.
    """
    _ACTION_RE = re.compile(
        r"""Action\s*\d*:\s*(.*?)\s*\n        # tool name
            Action\ Input\s*\d*:\s*([\s\S]*?) # tool input (non-greedy)
            (?=\nAction|\nObservation|\nFinal|\Z)  # stop at next block
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    def parse(self, text: str) -> Union[List[AgentAction], AgentFinish]:
        # ---------- collect all tool calls ----------
        matches = self._ACTION_RE.findall(text)
        if matches:
            return [
                AgentAction(
                    tool=tool.strip(),
                    tool_input=tool_input.strip().strip('"'),
                    log=text,
                )
                for tool, tool_input in matches
            ]

        # ---------- or a final answer ----------
        # final = re.search(r"(?:Final Answer|Final Thought):\s*(.*)", text, flags=re.IGNORECASE | re.DOTALL)
        final = re.search(r"(?:Final Answer):\s*(.*)", text, flags=re.IGNORECASE | re.DOTALL)

        # final = re.search(r"(?i)Final Answer:\s*(?:\*{2})?\s*\n+(.*)", text, flags=re.IGNORECASE | re.DOTALL)   ###PushToQA 04-07-2025 fixed ** issue

        if final:
            return AgentFinish(
                return_values={"output": final.group(1).strip()}, log=text
            )

        # ---------- fallback if no Action or Final Answer but appears like final output ----------
        if text.strip():  # fallback: any non-empty text is assumed to be the final answer
            return AgentFinish(
                return_values={"output": text.strip()}, log=text
            )

        raise ValueError(f"Could not parse agent output: {text!r}")


# 2️⃣  ───── Executor that really runs in parallel ────────────────
class ParallelAgentExecutor(AgentExecutor):
    async def _aiter_next_step(self, name_to_tool_map, inputs):
        llm_output = await self.agent.aplan(
            intermediate_steps=self.intermediate_steps, **inputs
        )
        parsed = self.output_parser.parse(llm_output)

        # ----- run *all* Actions concurrently -----
        if isinstance(parsed, list):
            coros = [self._aperform_agent_action(name_to_tool_map, a) for a in parsed]
            results = await asyncio.gather(*coros)

            for a, r in zip(parsed, results):
                self.intermediate_steps.append((a, r))
                yield f"Observation: {r}\n"

        elif isinstance(parsed, AgentFinish):
            self.intermediate_steps.append(
                (parsed, parsed.return_values["output"])
            )
            yield f"Final Answer: {parsed.return_values['output']}\n"


# 3️⃣  ───── Helper to build the agent + prompt ───────────────────
def get_react_agent(llm, tools, system_prompt, verbose=False):
    """Helper function for creating agent executor"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="conversation_history", optional=True),
        HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
        AIMessagePromptTemplate(
            prompt=PromptTemplate(input_variables=['agent_scratchpad'], template='{agent_scratchpad}')),
    ])
    agent = create_react_agent(llm, tools, prompt, output_parser=MultiActionOutputParser())

    executor = ParallelAgentExecutor(agent=agent,
                                     tools=tools,
                                     verbose=verbose,
                                     stream_runnable=True,
                                     handle_parsing_errors=True,
                                     max_iterations=20,
                                     return_intermediate_steps=True,
                                     )
    return executor


In [17]:
# 4. Graphiti setup (real temporal memory)
def setup_graphiti():
    # version-sensitive Graphiti init isolated here
    graphiti = Graphiti(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)
    # build indices/constraints once (safe to re-run)
    asyncio.get_event_loop().run_until_complete(graphiti.build_indices_and_constraints())
    STATE["graphiti"] = graphiti
    print("Graphiti ready.")
    return graphiti

In [18]:
# 5. HybridRetriever setup (Neo4j GraphRAG)
def setup_hybrid_retriever():
    # version-sensitive GraphRAG calls isolated here
    embedder = GraphRAGEmbeddings(model="text-embedding-3-small")
    retriever = HybridRetriever(
        driver=STATE["driver"],
        vector_index_name=VECTOR_INDEX,
        fulltext_index_name=FULLTEXT_INDEX,
        embedder=embedder,
    )
    STATE["retriever"] = retriever
    print("HybridRetriever ready.")
    return retriever

In [19]:
# 6a. Retrieval function (used only through the tool)
def hybrid_movie_search(query: str) -> str:
    # version-sensitive retriever call isolated here
    retriever = STATE["retriever"]
    result = retriever.search(query_text=query, top_k=5)

    lines = []
    for item in result.items:
        text = getattr(item, "content", str(item))
        lines.append(f"- {text}")
    if not lines:
        return "No candidate movies found."
    return "Candidate movies:\n" + "\n".join(lines)

In [20]:
# 6b. Wrap HybridRetriever as a LangChain tool
def build_movie_tool():
    return Tool(
        name="HybridMovieSearch",
        func=hybrid_movie_search,
        description=(
            "Use this tool to retrieve movie candidates from the Neo4j movie "
            "knowledge graph using hybrid vector and keyword search. "
            "Input should be a short natural language movie query."
        ),
    )

In [ ]:
# 6c. Simple rule-based preference extractor
GENRES = ["sci-fi", "science fiction", "thriller", "horror", "comedy",
          "romance", "action", "drama", "fantasy", "documentary", "animation"]
MOODS = ["emotional", "funny", "dark", "uplifting", "scary", "relaxing", "intense"]

def extract_preferences(text: str) -> dict:
    t = text.lower()
    prefs = {"liked_genres": [], "disliked_genres": [],
             "liked_movies": [], "disliked_movies": [],
             "themes": [], "source_text": text}

    for g in GENRES:
        if g in t:
            # dislike if negation appears near the genre / message
            if re.search(r"(not|don'?t|dislike|hate|avoid|no)\b.*" + re.escape(g), t):
                prefs["disliked_genres"].append(g)
            else:
                prefs["liked_genres"].append(g)

    for m in MOODS:
        if m in t:
            prefs["themes"].append(m)
    if "something emotional" in t or "want something" in t:
        for m in MOODS:
            if m in t and m not in prefs["themes"]:
                prefs["themes"].append(m)

    # liked movies: "loved X", "liked X"
    for verb in ["loved", "love", "liked", "enjoyed"]:
        for mt in re.findall(verb + r"\s+([A-Z][\w'-]+(?:\s[A-Z][\w'-]+)*)", text):
            prefs["liked_movies"].append(mt.strip())

    # disliked movies: "hated X"
    for mt in re.findall(r"hated\s+([A-Z][\w'-]+(?:\s[A-Z][\w'-]+)*)", text):
        prefs["disliked_movies"].append(mt.strip())

    # dedupe
    for k, v in prefs.items():
        if isinstance(v, list):
            prefs[k] = list(dict.fromkeys(v))
    return prefs

In [ ]:
# 6d. Graphiti memory functions (real temporal memory)
def _run(coro):
    return asyncio.get_event_loop().run_until_complete(coro)

def add_conversation_episode(user_id: str, text: str):
    graphiti = STATE["graphiti"]
    _run(graphiti.add_episode(
        name=f"turn-{datetime.now(timezone.utc).isoformat()}",
        episode_body=text,
        source=EpisodeType.text,
        source_description=f"conversation with {user_id}",
        reference_time=datetime.now(timezone.utc),
        group_id=user_id,
    ))

def save_preferences_to_graphiti(user_id: str, prefs: dict, text: str):
    # store the raw conversation as an episode (Graphiti extracts + evolves facts)
    add_conversation_episode(user_id, text)

    # also add an explicit preference fact episode for clarity
    fact_lines = []
    if prefs["liked_genres"]:
        fact_lines.append("User likes genres: " + ", ".join(prefs["liked_genres"]))
    if prefs["disliked_genres"]:
        fact_lines.append("User dislikes genres: " + ", ".join(prefs["disliked_genres"]))
    if prefs["liked_movies"]:
        fact_lines.append("User liked movies: " + ", ".join(prefs["liked_movies"]))
    if prefs["disliked_movies"]:
        fact_lines.append("User disliked movies: " + ", ".join(prefs["disliked_movies"]))
    if prefs["themes"]:
        fact_lines.append("User wants mood/themes: " + ", ".join(prefs["themes"]))

    if fact_lines:
        add_conversation_episode(user_id, ". ".join(fact_lines) + ".")
    print("Preferences saved to Graphiti.")

def load_active_preferences(user_id: str):
    graphiti = STATE["graphiti"]
    # temporal search: current valid preference facts for this user
    results = _run(graphiti.search(
        query="user movie preferences liked and disliked genres and movies",
        group_ids=[user_id],
        num_results=10,
    ))
    facts = [r.fact for r in results]
    return facts

In [ ]:
# 7. Build ReAct agent
def build_agent_prompt():
    template = """You are a concise movie recommendation assistant.
Use stored user preferences if provided. Use the HybridMovieSearch tool to get
movie candidates. Do not invent movie candidates without using the tool.

You have access to the following tools:
{tools}

Use this format:
Question: the input question
Thought: think about what to do
Action: the action to take, one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: top 3 recommendations, each with one short reason

Question: {input}
{agent_scratchpad}"""
    return PromptTemplate.from_template(template)

def build_react_agent():
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [build_movie_tool()]
    prompt = build_agent_prompt()
    agent = create_react_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools,
                             verbose=True, handle_parsing_errors=True,
                             max_iterations=4)
    STATE["agent"] = executor
    print("ReAct agent ready.")
    return executor

In [ ]:
# 8. Turn runner + preference display
def run_turn(user_id: str, user_message: str):
    prefs_facts = load_active_preferences(user_id)
    memory_text = "; ".join(prefs_facts) if prefs_facts else "none"
    print(f"\n--- Turn ---\nUser: {user_message}\nKnown preferences: {memory_text}")

    agent_input = (
        f"User request: {user_message}\n"
        f"Known user preferences: {memory_text}\n"
        "Recommend movies using the HybridMovieSearch tool."
    )
    result = STATE["agent"].invoke({"input": agent_input})
    answer = result["output"]
    print("\nAssistant:\n" + answer)
    return answer

def show_preferences(user_id: str):
    facts = load_active_preferences(user_id)
    df = pd.DataFrame({"active_preferences": facts if facts else ["(none yet)"]})
    print("\nActive preferences in Graphiti:")
    print(df.to_string(index=False))
    return df

# 8b. Conversational turn: check memory -> ask follow-up if needed -> update graph -> answer
def converse(user_id: str, user_message: str, ask_fn=input):
    # 1. try to fetch existing preferences from the living graph
    facts = load_active_preferences(user_id)

    # 2. if nothing useful is stored, ask one short follow-up and learn
    if not facts:
        print("Assistant: I don't know your taste yet. "
              "What genres do you usually like, and anything you want to avoid?")
        reply = ask_fn("You: ")
        prefs = extract_preferences(reply)
        save_preferences_to_graphiti(user_id, prefs, reply)  # update living graph
        facts = load_active_preferences(user_id)
    else:
        print(f"Assistant: Welcome back! I remember: {'; '.join(facts)}")
        # still learn from anything new in this message
        prefs = extract_preferences(user_message)
        if any(prefs[k] for k in ("liked_genres", "disliked_genres",
                                  "liked_movies", "disliked_movies", "themes")):
            save_preferences_to_graphiti(user_id, prefs, user_message)
            facts = load_active_preferences(user_id)

    # 3. answer through the ReAct agent using the tool
    return run_turn(user_id, user_message)

In [ ]:
# 9. Full demo end-to-end
# initialize everything
setup_neo4j()
setup_graphiti()
setup_hybrid_retriever()
build_react_agent()

# --- Turn 1 ---
turn1_msg = "What should I watch today?"
existing = load_active_preferences(USER_ID)
if not existing:
    print("Assistant: What genres do you usually like, and anything you want to avoid?")
    follow_up = "I like sci-fi and thrillers, I do not like horror, and I loved Interstellar"
    print(f"User: {follow_up}")
    prefs = extract_preferences(follow_up)
    save_preferences_to_graphiti(USER_ID, prefs, follow_up)

run_turn(USER_ID, turn1_msg)

# --- Turn 2 (uses memory) ---
run_turn(USER_ID, "Suggest something for tonight")

# --- Show active preferences ---
show_preferences(USER_ID)

In [ ]:
# 10. Interactive chat window (ipywidgets)
import ipywidgets as widgets
from IPython.display import display

def _redirect_print(out_widget, fn, *args, **kwargs):
    # capture prints from converse/agent into the output widget
    import contextlib, io
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        result = fn(*args, **kwargs)
    with out_widget:
        print(buf.getvalue())
    return result

def launch_chat_ui(user_id: str):
    out = widgets.Output(layout={"border": "1px solid #ddd", "height": "320px",
                                 "overflow": "auto", "padding": "6px"})
    box = widgets.Text(placeholder="Ask e.g. 'What should I watch today?'",
                       layout=widgets.Layout(width="80%"))
    send = widgets.Button(description="Send", button_style="primary")

    ui_state = {"awaiting_prefs": False}

    def handle(_=None):
        msg = box.value.strip()
        if not msg:
            return
        box.value = ""
        with out:
            print(f"\nYou: {msg}")
        if msg.lower() in {"quit", "exit", "bye"}:
            with out:
                print("Assistant: Enjoy your movie!")
            return

        # If we asked for preferences last turn, this message IS the answer.
        if ui_state["awaiting_prefs"]:
            prefs = extract_preferences(msg)
            _redirect_print(out, save_preferences_to_graphiti, user_id, prefs, msg)
            ui_state["awaiting_prefs"] = False
            _redirect_print(out, run_turn, user_id, msg)
            return

        # Normal turn: check living graph for stored preferences.
        facts = _redirect_print(out, load_active_preferences, user_id)
        if not facts:
            with out:
                print("Assistant: I don't know your taste yet. "
                      "What genres do you like, and anything to avoid?")
            ui_state["awaiting_prefs"] = True
            return

        with out:
            print(f"Assistant: I remember: {'; '.join(facts)}")
        # learn anything new in this message, then recommend
        new = extract_preferences(msg)
        if any(new[k] for k in ("liked_genres", "disliked_genres",
                                "liked_movies", "disliked_movies", "themes")):
            _redirect_print(out, save_preferences_to_graphiti, user_id, new, msg)
        _redirect_print(out, run_turn, user_id, msg)

    send.on_click(handle)
    box.on_submit(handle)  # Enter key

    display(widgets.VBox([out, widgets.HBox([box, send])]))
    with out:
        print("Movie assistant ready. Type a message and press Enter.")

launch_chat_ui(USER_ID)